In [1]:
import polars as pl
import pandas as pd
import pyranges as pr

/opt/modules/i12g/anaconda/envs/sl-ukg/lib/python3.12/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
anno_sub = pl.read_parquet('/s/project/deeprvat/ukb_gym/var_lists/ukbgym_variants_snps_dms_251001.parquet')
anno_sub

id,gene_id,chrom,pos,ref,alt
str,str,str,i32,str,str
"""chr1:1320456:C:T""","""ENSG00000127054""","""chr1""",1320456,"""C""","""T"""
"""chr1:1320460:T:C""","""ENSG00000127054""","""chr1""",1320460,"""T""","""C"""
"""chr1:1320468:C:G""","""ENSG00000127054""","""chr1""",1320468,"""C""","""G"""
"""chr1:1320476:G:C""","""ENSG00000127054""","""chr1""",1320476,"""G""","""C"""
"""chr1:1320479:G:A""","""ENSG00000127054""","""chr1""",1320479,"""G""","""A"""
…,…,…,…,…,…
"""chr22:42912069:T:C""","""ENSG00000100266""","""chr22""",42912069,"""T""","""C"""
"""chr22:42912072:G:A""","""ENSG00000100266""","""chr22""",42912072,"""G""","""A"""
"""chr22:42912072:G:C""","""ENSG00000100266""","""chr22""",42912072,"""G""","""C"""


In [3]:
# Path to your GENCODE GTF file
gtf_path = "/s/project/deeprvat/ukb_gym/gencode/gencode.v49.annotation.gtf.gz"

# Read GTF and filter for transcripts
gencode_pr = pr.read_gtf(gtf_path)
gencode_pr

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_type,...,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
0,chr1,HAVANA,gene,11120,24894,.,+,.,ENSG00000290825.2,lncRNA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,HAVANA,transcript,11120,14413,.,+,.,ENSG00000290825.2,lncRNA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,HAVANA,exon,11120,11211,.,+,.,ENSG00000290825.2,lncRNA,...,1,ENSE00004248723.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,HAVANA,exon,12009,12227,.,+,.,ENSG00000290825.2,lncRNA,...,2,ENSE00004248735.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,HAVANA,exon,12612,12721,.,+,.,ENSG00000290825.2,lncRNA,...,3,ENSE00003582793.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7750149,chrY,HAVANA,exon,57214349,57214397,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,1,ENSE00004015123.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,NaN,NaN,NaN
7750150,chrY,HAVANA,exon,57213879,57213964,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,2,ENSE00004015124.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,NaN,NaN,NaN
7750151,chrY,HAVANA,exon,57213525,57213602,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,3,ENSE00004015125.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,NaN,NaN,NaN
7750152,chrY,HAVANA,exon,57213203,57213357,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,4,ENSE00004015126.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,NaN,NaN,NaN


In [4]:
gencode_pr.df['Feature'].unique()

array(['gene', 'transcript', 'exon', 'CDS', 'start_codon', 'stop_codon',
       'UTR', 'Selenocysteine'], dtype=object)

In [5]:
gpl = pl.from_pandas(gencode_pr.df)

filt_gpl = gpl.filter(
    (pl.col('transcript_type') == 'protein_coding') |
    (pl.col('transcript_type').str.contains('pseudogene')) |
    (pl.col('Feature') == 'CDS')
).filter(
    pl.col('Feature') != 'transcript'
)
filt_gpl

Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
cat,str,str,i64,i64,str,cat,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""chr1""","""HAVANA""","""exon""",12009,12057,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""1""","""ENSE00001948541.1""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null
"""chr1""","""HAVANA""","""exon""",12178,12227,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""2""","""ENSE00001671638.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null
"""chr1""","""HAVANA""","""exon""",12612,12697,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""3""","""ENSE00001758273.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null
"""chr1""","""HAVANA""","""exon""",12974,13052,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""4""","""ENSE00001799933.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null
"""chr1""","""HAVANA""","""exon""",13220,13374,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""5""","""ENSE00001746346.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""","""HAVANA""","""exon""",57214349,57214397,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""1""","""ENSE00004015123.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null
"""chrY""","""HAVANA""","""exon""",57213879,57213964,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""2""","""ENSE00004015124.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null
"""chrY""","""HAVANA""","""exon""",57213525,57213602,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""3""","""ENSE00004015125.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null


In [6]:
filt_gpl['transcript_type'].value_counts(sort=True)

transcript_type,count
str,u32
"""protein_coding""",5550761
"""nonsense_mediated_decay""",115666
"""processed_pseudogene""",10974
"""transcribed_unprocessed_pseudo…",9036
"""unprocessed_pseudogene""",4613
…,…
"""TR_D_gene""",5
"""TR_J_pseudogene""",4
"""IG_J_pseudogene""",3


In [7]:
filt_gpl['gene_type'].value_counts(sort=True)

gene_type,count
str,u32
"""protein_coding""",5667691
"""processed_pseudogene""",10974
"""transcribed_unprocessed_pseudo…",9036
"""unprocessed_pseudogene""",4613
"""transcribed_unitary_pseudogene""",1485
…,…
"""TR_D_gene""",5
"""TR_J_pseudogene""",4
"""IG_J_pseudogene""",3


In [8]:
anti_filt_gpl = gpl.filter(
    (pl.col('transcript_type') != 'protein_coding') &
    (~pl.col('transcript_type').str.contains('pseudogene')) & 
    (pl.col('Feature') != 'CDS')
)

anti_filt_gpl

Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl
cat,str,str,i64,i64,str,cat,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""chr1""","""HAVANA""","""transcript""",11120,14413,""".""","""+""",""".""","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""","""2""","""TAGENE""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""",null,null,null,null,null,null,null,null,null,null
"""chr1""","""HAVANA""","""exon""",11120,11211,""".""","""+""",""".""","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""","""2""","""TAGENE""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""","""1""","""ENSE00004248723.1""",null,null,null,null,null,null,null,null
"""chr1""","""HAVANA""","""exon""",12009,12227,""".""","""+""",""".""","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""","""2""","""TAGENE""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""","""2""","""ENSE00004248735.1""",null,null,null,null,null,null,null,null
"""chr1""","""HAVANA""","""exon""",12612,12721,""".""","""+""",""".""","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""","""2""","""TAGENE""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""","""3""","""ENSE00003582793.1""",null,null,null,null,null,null,null,null
"""chr1""","""HAVANA""","""exon""",13452,14413,""".""","""+""",""".""","""ENSG00000290825.2""","""lncRNA""","""DDX11L16""","""2""","""TAGENE""","""ENST00000832824.1""","""lncRNA""","""DDX11L16-260""","""4""","""ENSE00004248730.1""",null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""","""HAVANA""","""exon""",57213879,57213964,""".""","""-""",""".""","""ENSG00000310542.1""","""lncRNA""","""ENSG00000310542""","""2""","""TAGENE""","""ENST00000972826.1""","""lncRNA""","""ENST00000972826""","""3""","""ENSE00004471633.1""",null,null,null,null,null,null,null,null
"""chrY""","""HAVANA""","""exon""",57212922,57213125,""".""","""-""",""".""","""ENSG00000310542.1""","""lncRNA""","""ENSG00000310542""","""2""","""TAGENE""","""ENST00000972826.1""","""lncRNA""","""ENST00000972826""","""4""","""ENSE00004471649.1""",null,null,null,null,null,null,null,null
"""chrY""","""HAVANA""","""transcript""",57213905,57214730,""".""","""-""",""".""","""ENSG00000310542.1""","""lncRNA""","""ENSG00000310542""","""2""","""TAGENE""","""ENST00000972827.1""","""lncRNA""","""ENST00000972827""",null,null,null,null,null,null,null,null,null,null


In [9]:
filt_gpl = filt_gpl.with_columns(
    Gene = pl.col('gene_id').str.split('.').list.get(0)
)
filt_gpl

Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_type,gene_name,level,tag,transcript_id,transcript_type,transcript_name,exon_number,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl,Gene
cat,str,str,i64,i64,str,cat,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""chr1""","""HAVANA""","""exon""",12009,12057,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""1""","""ENSE00001948541.1""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null,"""ENSG00000223972"""
"""chr1""","""HAVANA""","""exon""",12178,12227,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""2""","""ENSE00001671638.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null,"""ENSG00000223972"""
"""chr1""","""HAVANA""","""exon""",12612,12697,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""3""","""ENSE00001758273.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null,"""ENSG00000223972"""
"""chr1""","""HAVANA""","""exon""",12974,13052,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""4""","""ENSE00001799933.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null,"""ENSG00000223972"""
"""chr1""","""HAVANA""","""exon""",13220,13374,""".""","""+""",""".""","""ENSG00000223972.6""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""2""","""GENCODE_Primary""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""","""5""","""ENSE00001746346.2""","""NA""","""OTTHUMT00000002844.2""","""HGNC:37102""","""OTTHUMG00000000961.2""","""PGO:0000019""",null,null,null,"""ENSG00000223972"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""","""HAVANA""","""exon""",57214349,57214397,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""1""","""ENSE00004015123.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null,"""ENSG00000292371"""
"""chrY""","""HAVANA""","""exon""",57213879,57213964,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""2""","""ENSE00004015124.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null,"""ENSG00000292371"""
"""chrY""","""HAVANA""","""exon""",57213525,57213602,""".""","""-""",""".""","""ENSG00000292371.1""","""unprocessed_pseudogene""","""DDX11L16""","""2""","""GENCODE_Primary""","""ENST00000711270.1""","""unprocessed_pseudogene""","""DDX11L16-286""","""3""","""ENSE00004015125.1""","""NA""","""OTTHUMT00000058841.1""","""HGNC:37115""","""OTTHUMG00000022678.1""","""PGO:0000005""",null,null,null,"""ENSG00000292371"""


In [10]:
filt_gpl['Feature'].value_counts(sort=True)

Feature,count
str,u32
"""exon""",2410438
"""CDS""",2283012
"""UTR""",607057
"""start_codon""",203710
"""stop_codon""",192747
"""Selenocysteine""",119


In [11]:
gencode_pr_cds = pr.PyRanges(filt_gpl.to_pandas())
gencode_pr_cds

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_type,...,exon_id,transcript_support_level,havana_transcript,hgnc_id,havana_gene,ont,protein_id,ccdsid,artif_dupl,Gene
0,chr1,HAVANA,exon,12009,12057,.,+,.,ENSG00000223972.6,transcribed_unprocessed_pseudogene,...,ENSE00001948541.1,NA,OTTHUMT00000002844.2,HGNC:37102,OTTHUMG00000000961.2,PGO:0000019,None,None,None,ENSG00000223972
1,chr1,HAVANA,exon,12178,12227,.,+,.,ENSG00000223972.6,transcribed_unprocessed_pseudogene,...,ENSE00001671638.2,NA,OTTHUMT00000002844.2,HGNC:37102,OTTHUMG00000000961.2,PGO:0000019,None,None,None,ENSG00000223972
2,chr1,HAVANA,exon,12612,12697,.,+,.,ENSG00000223972.6,transcribed_unprocessed_pseudogene,...,ENSE00001758273.2,NA,OTTHUMT00000002844.2,HGNC:37102,OTTHUMG00000000961.2,PGO:0000019,None,None,None,ENSG00000223972
3,chr1,HAVANA,exon,12974,13052,.,+,.,ENSG00000223972.6,transcribed_unprocessed_pseudogene,...,ENSE00001799933.2,NA,OTTHUMT00000002844.2,HGNC:37102,OTTHUMG00000000961.2,PGO:0000019,None,None,None,ENSG00000223972
4,chr1,HAVANA,exon,13220,13374,.,+,.,ENSG00000223972.6,transcribed_unprocessed_pseudogene,...,ENSE00001746346.2,NA,OTTHUMT00000002844.2,HGNC:37102,OTTHUMG00000000961.2,PGO:0000019,None,None,None,ENSG00000223972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5697078,chrY,HAVANA,exon,57214349,57214397,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,ENSE00004015123.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,None,None,None,ENSG00000292371
5697079,chrY,HAVANA,exon,57213879,57213964,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,ENSE00004015124.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,None,None,None,ENSG00000292371
5697080,chrY,HAVANA,exon,57213525,57213602,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,ENSE00004015125.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,None,None,None,ENSG00000292371
5697081,chrY,HAVANA,exon,57213203,57213357,.,-,.,ENSG00000292371.1,unprocessed_pseudogene,...,ENSE00004015126.1,NA,OTTHUMT00000058841.1,HGNC:37115,OTTHUMG00000022678.1,PGO:0000005,None,None,None,ENSG00000292371


In [12]:
gencode_coding_regions = gencode_pr_cds.merge()
gencode_coding_regions

,Chromosome,Start,End
0,chr1,12009,12057
1,chr1,12178,12227
2,chr1,12612,12697
3,chr1,12974,13052
4,chr1,13220,13374
...,...,...,...
269751,chrY,57212183,57213125
269752,chrY,57213203,57213357
269753,chrY,57213525,57213602
269754,chrY,57213879,57213964


## Annotate UKBBGym variants

In [13]:
anno = (
    pl.read_parquet(
        '/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_250928.parquet', 
        columns=['id', 'chrom', 'pos', 'ref', 'alt', 'gene']
    )
    .with_columns(
        Chromosome = pl.col('chrom'),
        Start = pl.col('pos') - 1,
        End = pl.col('pos') - 1 + pl.col('ref').str.len_chars()
    )
    .select(['id', 'Chromosome', 'Start', 'End', 'ref', 'alt', 'gene'])
)

anno_pr = pr.PyRanges(anno.to_pandas())

anno_pr

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:214626910:A:G,chr1,214626909,214626910,A,G,ENSG00000117724
1,chr1:21556831:T:G,chr1,21556830,21556831,T,G,ENSG00000162551
2,chr1:21569280:C:A,chr1,21569279,21569280,C,A,ENSG00000162551
3,chr1:21556795:T:C,chr1,21556794,21556795,T,C,ENSG00000162551
4,chr1:214648630:A:G,chr1,214648629,214648630,A,G,ENSG00000117724
...,...,...,...,...,...,...,...
25373672,chr22:17093668:AGGTAAC:A,chr22,17093667,17093674,AGGTAAC,A,ENSG00000177663
25373673,chr22:36919903:GCATA:G,chr22,36919902,36919907,GCATA,G,ENSG00000100368
25373674,chr22:41102783:GT:G,chr22,41102782,41102784,GT,G,ENSG00000100393
25373675,chr22:24508167:ATC:A,chr22,24508166,24508169,ATC,A,ENSG00000100024


In [14]:
anno_pr_nc = anno_pr.overlap(gencode_coding_regions, invert=True)
anno_pr_nc

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:214626910:A:G,chr1,214626909,214626910,A,G,ENSG00000117724
1,chr1:21556831:T:G,chr1,21556830,21556831,T,G,ENSG00000162551
2,chr1:21569280:C:A,chr1,21569279,21569280,C,A,ENSG00000162551
3,chr1:21556795:T:C,chr1,21556794,21556795,T,C,ENSG00000162551
4,chr1:214648630:A:G,chr1,214648629,214648630,A,G,ENSG00000117724
...,...,...,...,...,...,...,...
23367783,chr22:41148660:AATTTGTGTT:A,chr22,41148659,41148669,AATTTGTGTT,A,ENSG00000100393
23367784,chr22:36919903:GCATA:G,chr22,36919902,36919907,GCATA,G,ENSG00000100368
23367785,chr22:41102783:GT:G,chr22,41102782,41102784,GT,G,ENSG00000100393
23367786,chr22:24508167:ATC:A,chr22,24508166,24508169,ATC,A,ENSG00000100024


In [15]:
anno_pr_coding = anno_pr.overlap(gencode_coding_regions)
print('Coding variants:', len(anno_pr_coding))

anno_pr_coding
# anno_pr_coding.df['consequence'].value_counts()

Coding variants: 2005889


,id,Chromosome,Start,End,ref,alt,gene
0,chr1:198320819:T:C,chr1,198320818,198320819,T,C,ENSG00000151414
1,chr1:198320797:G:C,chr1,198320796,198320797,G,C,ENSG00000151414
2,chr1:214646777:A:G,chr1,214646776,214646777,A,G,ENSG00000117724
3,chr1:198256393:C:T,chr1,198256392,198256393,C,T,ENSG00000151414
4,chr1:214655398:A:G,chr1,214655397,214655398,A,G,ENSG00000117724
...,...,...,...,...,...,...,...
2005884,chr22:40042022:CTATT:C,chr22,40042021,40042026,CTATT,C,ENSG00000100354
2005885,chr22:28738166:CTG:C,chr22,28738165,28738168,CTG,C,ENSG00000183765
2005886,chr22:40040232:CAG:C,chr22,40040231,40040234,CAG,C,ENSG00000100354
2005887,chr22:36940069:GTA:G,chr22,36940068,36940071,GTA,G,ENSG00000100368


In [16]:
set(anno_pr_nc.df['id'].to_list()).intersection(set(anno_pr_coding.df['id'].to_list()))

set()

## Merge back with annotations df

In [17]:
anno_pl_nc = pl.from_pandas(anno_pr_nc.df)

anno_pl_nc = anno_pl_nc.with_columns(
    gencode_non_coding = pl.lit(True),
    Chromosome = pl.col('Chromosome').cast(pl.Utf8)
)

anno_pl_nc

id,Chromosome,Start,End,ref,alt,gene,gencode_non_coding
str,str,i64,i64,str,str,str,bool
"""chr1:214626910:A:G""","""chr1""",214626909,214626910,"""A""","""G""","""ENSG00000117724""",true
"""chr1:21556831:T:G""","""chr1""",21556830,21556831,"""T""","""G""","""ENSG00000162551""",true
"""chr1:21569280:C:A""","""chr1""",21569279,21569280,"""C""","""A""","""ENSG00000162551""",true
"""chr1:21556795:T:C""","""chr1""",21556794,21556795,"""T""","""C""","""ENSG00000162551""",true
"""chr1:214648630:A:G""","""chr1""",214648629,214648630,"""A""","""G""","""ENSG00000117724""",true
…,…,…,…,…,…,…,…
"""chr22:41148660:AATTTGTGTT:A""","""chr22""",41148659,41148669,"""AATTTGTGTT""","""A""","""ENSG00000100393""",true
"""chr22:36919903:GCATA:G""","""chr22""",36919902,36919907,"""GCATA""","""G""","""ENSG00000100368""",true
"""chr22:41102783:GT:G""","""chr22""",41102782,41102784,"""GT""","""G""","""ENSG00000100393""",true


In [18]:
a = pl.scan_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_250928.parquet')
a.collect_schema().names()

['chrom',
 'pos',
 'ref',
 'alt',
 'id',
 'col',
 'gene',
 'distance',
 'polyphen',
 'cadd_raw',
 'am_pathogenicity',
 'spliceai_delta_score',
 'consequence_3_prime_utr_variant',
 'consequence_5_prime_utr_variant',
 'consequence_nmd_transcript_variant',
 'consequence_coding_sequence_variant',
 'consequence_downstream_gene_variant',
 'consequence_frameshift_variant',
 'consequence_inframe_deletion',
 'consequence_inframe_insertion',
 'consequence_intergenic_variant',
 'consequence_intron_variant',
 'consequence_missense_variant',
 'consequence_non_coding_transcript_exon_variant',
 'consequence_non_coding_transcript_variant',
 'consequence_protein_altering_variant',
 'consequence_splice_acceptor_variant',
 'consequence_splice_donor_5th_base_variant',
 'consequence_splice_donor_region_variant',
 'consequence_splice_donor_variant',
 'consequence_splice_polypyrimidine_tract_variant',
 'consequence_splice_region_variant',
 'consequence_start_lost',
 'consequence_start_retained_variant',
 'co

In [19]:
vep_coding = [
    'consequence_coding_sequence_variant',
    'consequence_nmd_transcript_variant',
    'consequence_frameshift_variant',
    'consequence_missense_variant',
    'consequence_protein_altering_variant',
    'consequence_splice_acceptor_variant',
    'consequence_splice_donor_5th_base_variant',
    'consequence_splice_donor_region_variant',
    'consequence_splice_donor_variant',
    'consequence_splice_polypyrimidine_tract_variant',
    'consequence_splice_region_variant',
    'consequence_start_lost',
    'consequence_start_retained_variant',
    'consequence_stop_gained',
    'consequence_stop_lost',
    'consequence_stop_retained_variant',
    'consequence_synonymous_variant',
]

vep_nc = [
    'consequence_3_prime_utr_variant',
    'consequence_5_prime_utr_variant',
    'consequence_downstream_gene_variant',
    'consequence_inframe_deletion',
    'consequence_inframe_insertion',
    'consequence_intergenic_variant',
    'consequence_intron_variant',
    'consequence_non_coding_transcript_exon_variant',
    'consequence_non_coding_transcript_variant',
    'consequence_upstream_gene_variant'
]

In [ ]:
anno = (
    pl.scan_parquet(
        '/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_250928.parquet',
    )
    .with_columns(
        vep_coding = pl.sum_horizontal(vep_coding) > 0,
    )
    .collect()
)

anno

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,dbscsnv-ada_score_is_nan,dbscsnv-rf_score_is_nan,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,vep_coding
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293,false
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965,false
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965,false
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978,false
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null,false
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.0,0.095799,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,0,0,1,1

In [21]:
anno_pl = anno.join(anno_pl_nc[['id', 'gene', 'gencode_non_coding']], on=['id', 'gene'], how='left').with_columns(pl.col("gencode_non_coding").fill_null(False))

anno_pl = anno_pl.with_columns(
    vep_gencode_non_coding = (pl.col('vep_coding')==False) & (pl.col('gencode_non_coding')==True)
)

anno_pl

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,vep_coding,gencode_non_coding,vep_gencode_non_coding
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool,bool,bool
"""chr2""",169197167,"""T""","""G""","""chr2:169197167:T:G""",12335303,"""ENSG00000081479""",0,0.0,0.466688,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",165367,0.002291,-0.006063,0.0,0.0,0.0,0.153293,false,true,true
"""chr2""",169204292,"""A""","""T""","""chr2:169204292:A:T""",12338123,"""ENSG00000081479""",0,0.0,0.104863,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",158242,0.011141,-0.026342,0.0,0.0,0.0,1.684965,false,true,true
"""chr2""",169030531,"""C""","""T""","""chr2:169030531:C:T""",12294336,"""ENSG00000073734""",0,0.0,0.326902,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",793,0.006871,-0.005501,0.0,0.0,0.0,-1.212965,false,true,true
"""chr2""",168910825,"""T""","""G""","""chr2:168910825:T:G""",12247951,"""ENSG00000073734""",4673,0.0,0.554549,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",120499,0.002667,-0.019354,0.0,0.0,0.0,-1.479978,false,true,true
"""chr2""",168969544,"""T""","""C""","""chr2:168969544:T:C""",12270653,"""ENSG00000073734""",0,0.0,1.464585,0.0607,0.02,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",61780,-0.009066,-0.053713,0.0,0.0,0.0,-2.904763,true,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",34974609,"""ACTTCT""","""A""","""chr15:34974609:ACTTCT:A""",66849879,"""ENSG00000198146""",3727,0.0,0.681056,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,34988287,"""-""",9948,"""ZNF770""",13678,null,null,0.0,0.0,0.0,null,false,true,true
"""chr9""",132358570,"""C""","""A""","""chr9:132358570:C:A""",48217034,"""ENSG00000107290""",3584,0.0,0.095799,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [ ]:
# anno_pl.write_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass394genes_olink371genes_variants_union_annotated_genocode_251031.parquet')


In [23]:
anno_pl.filter(
    (pl.col('gencode_non_coding') == True) &
    (pl.col('vep_gencode_non_coding') == False)
)

chrom,pos,ref,alt,id,col,gene,distance,polyphen,cadd_raw,am_pathogenicity,spliceai_delta_score,consequence_3_prime_utr_variant,consequence_5_prime_utr_variant,consequence_nmd_transcript_variant,consequence_coding_sequence_variant,consequence_downstream_gene_variant,consequence_frameshift_variant,consequence_inframe_deletion,consequence_inframe_insertion,consequence_intergenic_variant,consequence_intron_variant,consequence_missense_variant,consequence_non_coding_transcript_exon_variant,consequence_non_coding_transcript_variant,consequence_protein_altering_variant,consequence_splice_acceptor_variant,consequence_splice_donor_5th_base_variant,consequence_splice_donor_region_variant,consequence_splice_donor_variant,consequence_splice_polypyrimidine_tract_variant,consequence_splice_region_variant,consequence_start_lost,consequence_start_retained_variant,consequence_stop_gained,consequence_stop_lost,consequence_stop_retained_variant,…,remapoverlaptf_is_nan,remapoverlapcl_is_nan,esmscoremissense_is_nan,esmscoreinframe_is_nan,esmscoreframeshift_is_nan,regseq0_is_nan,regseq1_is_nan,regseq2_is_nan,regseq3_is_nan,regseq4_is_nan,regseq5_is_nan,regseq6_is_nan,regseq7_is_nan,aparent2_is_nan,zoopriphylop_is_nan,zooverphylop_is_nan,zoorocc_is_nan,zoouce_is_nan,roulette-filter_is_nan,roulette-mr_is_nan,roulette-ar_is_nan,cadd_rawscore_is_nan,cadd_phred_is_nan,TSS,Strand,gene_length,gene_name,dist_to_tss,abexp_max,abexp_min,promoterai_abs,promoterai_over,promoterai_under,gpn_star_llr_calibrated_mean,vep_coding,gencode_non_coding,vep_gencode_non_coding
str,i32,str,str,str,i32,str,i32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i64,cat,i64,str,i64,f64,f64,f32,f32,f32,f64,bool,bool,bool
"""chr2""",169220575,"""A""","""G""","""chr2:169220575:A:G""",12344233,"""ENSG00000081479""",0,0.0,0.900946,0.0,0.04,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,…,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169362534,"""-""",235427,"""LRP2""",141959,-0.000122,-0.064903,0.0,0.0,0.0,-3.372177,true,true,false
"""chr2""",165365131,"""A""","""G""","""chr2:165365131:A:G""",12176009,"""ENSG00000136531""",0,0.0,0.476776,0.0,0.01,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,…,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,165194992,"""+""",197319,"""SCN2A""",170139,-0.005303,-0.020578,0.0,0.0,0.0,-2.254203,true,true,false
"""chr2""",168958139,"""G""","""C""","""chr2:168958139:G:C""",12266117,"""ENSG00000073734""",0,0.0,-0.242806,0.0,0.15,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",73185,0.015046,-0.070216,0.0,0.0,0.0,-1.00055,true,true,false
"""chr2""",169013520,"""C""","""T""","""chr2:169013520:C:T""",12287878,"""ENSG00000073734""",0,0.0,1.800975,0.0,0.22,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",17804,-0.000569,-0.040002,0.0,0.0,0.0,-3.162808,true,true,false
"""chr2""",168964311,"""G""","""C""","""chr2:168964311:G:C""",12268622,"""ENSG00000073734""",0,0.0,3.187104,0.0,0.98,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,…,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,169031324,"""-""",115828,"""ABCB11""",67013,-0.022022,-0.370356,0.0,0.0,0.0,-10.263983,true,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr6""",16285842,"""CCGGGCCCTGGTG""","""C""","""chr6:16285842:CCGGGCCCTGGTG:C""",32232755,"""ENSG00000137198""",0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,16238586,"""+""",56964,"""GMPR""",47256,null,null,0.0,0.0,0.0,null,true,true,false
"""chr19""",38593223,"""GCT""","""G""","""chr19:38593223:GCT:G""",82731370,"""ENSG00000196218""",0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,0,1,0,0,0